In [7]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/processed/buildsafe_cleaned.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (102467, 22)

Columns:
['BIN', 'BORO', 'BLOCK', 'LOT', 'ISSUE_DATE', 'SEVERITY', 'VIOLATION_TYPE', 'RESPONDENT_CITY', 'RESPONDENT_ZIP', 'VIOLATION_DESCRIPTION', 'INFRACTION_CODE1', 'SECTION_LAW_DESCRIPTION1', 'ISSUE_DATE_DT', 'ISSUE_YEAR', 'ISSUE_MONTH', 'ISSUE_DAY', 'ISSUE_DAY_OF_WEEK', 'ISSUE_QUARTER', 'PREVIOUS_VIOLATION_COUNT', 'PREVIOUS_CLASS1_COUNT', 'PREVIOUS_CLASS2_COUNT', 'PREVIOUS_CLASS3_COUNT']


In [8]:
df["ISSUE_DATE_DT"] = pd.to_datetime(df["ISSUE_DATE_DT"])

print(df["ISSUE_DATE_DT"].dtype)
print(df["ISSUE_DATE_DT"].min())
print(df["ISSUE_DATE_DT"].max())

datetime64[us]
2025-01-01 00:00:00
2026-09-10 00:00:00


In [9]:
df = df.sort_values("ISSUE_DATE_DT").reset_index(drop=True)

print(
    df[["ISSUE_DATE_DT", "SEVERITY"]].head()
)

print(
    df[["ISSUE_DATE_DT", "SEVERITY"]].tail()
)

  ISSUE_DATE_DT   SEVERITY
0    2025-01-01  CLASS - 1
1    2025-01-01  CLASS - 1
2    2025-01-02  CLASS - 2
3    2025-01-02  CLASS - 1
4    2025-01-02  CLASS - 1
       ISSUE_DATE_DT   SEVERITY
102462    2026-09-10  CLASS - 2
102463    2026-09-10  CLASS - 1
102464    2026-09-10  CLASS - 1
102465    2026-09-10  CLASS - 2
102466    2026-09-10  CLASS - 1


In [10]:
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining period:")
print(
    train_df["ISSUE_DATE_DT"].min(),
    "→",
    train_df["ISSUE_DATE_DT"].max()
)

print("\nTesting period:")
print(
    test_df["ISSUE_DATE_DT"].min(),
    "→",
    test_df["ISSUE_DATE_DT"].max()
)

Training shape: (81973, 22)
Testing shape: (20494, 22)

Training period:
2025-01-01 00:00:00 → 2026-05-13 00:00:00

Testing period:
2026-05-13 00:00:00 → 2026-09-10 00:00:00


In [11]:
TARGET = "SEVERITY"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (81973, 21)
y_train: (81973,)
X_test: (20494, 21)
y_test: (20494,)


In [12]:
# ============================================================
# FINAL TEMPORAL SPLIT — DATE BOUNDARY
# ============================================================

# Use the date at the current 80% boundary
split_date = df.iloc[split_index]["ISSUE_DATE_DT"].normalize()

# Training = dates BEFORE split date
train_mask = df["ISSUE_DATE_DT"] < split_date

# Testing = dates ON/AFTER split date
test_mask = df["ISSUE_DATE_DT"] >= split_date

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()

print("Split date:", split_date)

print("\nTraining shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining period:")
print(train_df["ISSUE_DATE_DT"].min(), "→", train_df["ISSUE_DATE_DT"].max())

print("\nTesting period:")
print(test_df["ISSUE_DATE_DT"].min(), "→", test_df["ISSUE_DATE_DT"].max())

print("\nSame dates in both sets:",
      len(set(train_df["ISSUE_DATE_DT"]) &
          set(test_df["ISSUE_DATE_DT"])))

Split date: 2026-05-13 00:00:00

Training shape: (81903, 22)
Testing shape: (20564, 22)

Training period:
2025-01-01 00:00:00 → 2026-05-12 00:00:00

Testing period:
2026-05-13 00:00:00 → 2026-09-10 00:00:00

Same dates in both sets: 0


In [13]:
TARGET = "SEVERITY"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test: ", X_test.shape)
print("y_test: ", y_test.shape)

X_train: (81903, 21)
y_train: (81903,)
X_test:  (20564, 21)
y_test:  (20564,)


In [14]:
# ============================================================
# FEATURE TYPE & CARDINALITY INSPECTION
# ============================================================

feature_summary = pd.DataFrame({
    "Column": X_train.columns,
    "Dtype": X_train.dtypes.astype(str).values,
    "Unique": [X_train[col].nunique(dropna=False) for col in X_train.columns],
    "Missing": [X_train[col].isna().sum() for col in X_train.columns]
})

feature_summary

,Column,Dtype,Unique,Missing
0,BIN,str,33276,0
1,BORO,int64,5,0
2,BLOCK,str,9461,0
3,LOT,str,608,0
4,ISSUE_DATE,int64,497,0
5,VIOLATION_TYPE,str,12,0
6,RESPONDENT_CITY,str,1176,0
7,RESPONDENT_ZIP,str,1290,0
8,VIOLATION_DESCRIPTION,str,78228,0
9,INFRACTION_CODE1,str,313,0


In [15]:
# Show the summary sorted by number of unique values

feature_summary.sort_values("Unique")

,Column,Dtype,Unique,Missing
12,ISSUE_YEAR,int64,2,0
16,ISSUE_QUARTER,int64,4,0
20,PREVIOUS_CLASS3_COUNT,float64,5,0
1,BORO,int64,5,0
15,ISSUE_DAY_OF_WEEK,int64,7,0
5,VIOLATION_TYPE,str,12,0
13,ISSUE_MONTH,int64,12,0
18,PREVIOUS_CLASS1_COUNT,float64,24,0
19,PREVIOUS_CLASS2_COUNT,float64,27,0
14,ISSUE_DAY,int64,31,0


In [16]:
# ============================================================
# INVESTIGATE HIGH-CARDINALITY FEATURES
# ============================================================

for col in [
    "BLOCK",
    "LOT",
    "RESPONDENT_CITY",
    "RESPONDENT_ZIP",
    "SECTION_LAW_DESCRIPTION1"
]:
    print("\n" + "=" * 70)
    print(col)
    print("Unique values:", train_df[col].nunique())

    print("\nMost common values:")
    print(train_df[col].value_counts().head(10))


BLOCK
Unique values: 9461

Most common values:
BLOCK
UNKNOWN    410
2062.0      93
3319.0      85
2461.0      84
2179.0      79
2876.0      78
402.0       78
3263.0      78
1847.0      77
3250.0      75
Name: count, dtype: int64

LOT
Unique values: 608

Most common values:
LOT
1.0       7917
7501.0    1804
10.0      1301
11.0      1197
29.0      1153
7.0       1153
25.0      1153
5.0       1151
21.0      1146
6.0       1144
Name: count, dtype: int64

RESPONDENT_CITY
Unique values: 1176

Most common values:
RESPONDENT_CITY
BROOKLYN         23699
NEW YORK         10707
UNKNOWN           6586
BRONX             6476
JAMAICA           3361
FLUSHING          3201
STATEN ISLAND     3084
MANHATTAN         2794
QUEENS            1511
GREAT NECK        1135
Name: count, dtype: int64

RESPONDENT_ZIP
Unique values: 1290

Most common values:
RESPONDENT_ZIP
UNKNOWN    7163
11219      2053
11211      1633
11205      1322
11230      1227
11204      1208
11354      1188
10001      1126
11432      1107

In [17]:
# ============================================================
# TARGET DISTRIBUTION FOR HIGH-CARDINALITY FEATURES
# ============================================================

for col in [
    "BLOCK",
    "LOT",
    "RESPONDENT_CITY",
    "RESPONDENT_ZIP",
    "SECTION_LAW_DESCRIPTION1"
]:
    print("\n" + "=" * 70)
    print(col)

    result = pd.crosstab(
        train_df[col],
        train_df["SEVERITY"],
        normalize="index"
    )

    print(result.head(10))


BLOCK
SEVERITY  CLASS - 1  CLASS - 2  CLASS - 3
BLOCK                                    
1.0        0.111111   0.888889      0.000
10.0       0.666667   0.333333      0.000
100.0      0.000000   1.000000      0.000
1000.0     0.666667   0.333333      0.000
10002.0    0.000000   1.000000      0.000
10006.0    0.625000   0.000000      0.375
1001.0     1.000000   0.000000      0.000
10011.0    0.000000   1.000000      0.000
10012.0    0.166667   0.833333      0.000
10014.0    0.000000   0.800000      0.200

LOT
SEVERITY  CLASS - 1  CLASS - 2  CLASS - 3
LOT                                      
0.0        0.333333   0.666667   0.000000
1.0        0.289125   0.687508   0.023367
10.0       0.427364   0.534204   0.038432
100.0      0.292063   0.685714   0.022222
1000.0     0.000000   0.666667   0.333333
1001.0     1.000000   0.000000   0.000000
1002.0     0.666667   0.333333   0.000000
101.0      0.398148   0.537037   0.064815
1014.0     0.083333   0.916667   0.000000
1018.0     0.250000   

In [18]:
print("TRAINING TARGET")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTEST TARGET")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True).mul(100).round(2))

TRAINING TARGET
SEVERITY
CLASS - 2    46320
CLASS - 1    32213
CLASS - 3     3370
Name: count, dtype: int64
SEVERITY
CLASS - 2    56.55
CLASS - 1    39.33
CLASS - 3     4.11
Name: proportion, dtype: float64

TEST TARGET
SEVERITY
CLASS - 2    11287
CLASS - 1     8460
CLASS - 3      817
Name: count, dtype: int64
SEVERITY
CLASS - 2    54.89
CLASS - 1    41.14
CLASS - 3     3.97
Name: proportion, dtype: float64


In [19]:
drop_features = [
    "BIN",
    "BLOCK",
    "LOT",
    "ISSUE_DATE",
    "ISSUE_DATE_DT"
]

In [20]:
numeric_features = [
    "ISSUE_YEAR",
    "ISSUE_MONTH",
    "ISSUE_DAY",
    "ISSUE_DAY_OF_WEEK",
    "ISSUE_QUARTER",
    "PREVIOUS_VIOLATION_COUNT",
    "PREVIOUS_CLASS1_COUNT",
    "PREVIOUS_CLASS2_COUNT",
    "PREVIOUS_CLASS3_COUNT"
]

In [21]:
categorical_features = [
    "BORO",
    "VIOLATION_TYPE",
    "RESPONDENT_CITY",
    "RESPONDENT_ZIP",
    "INFRACTION_CODE1",
    "SECTION_LAW_DESCRIPTION1"
]

In [22]:
text_feature = "VIOLATION_DESCRIPTION"

In [23]:
print("Dropped:", len(drop_features))
print("Numerical:", len(numeric_features))
print("Categorical:", len(categorical_features))
print("Text:", text_feature)

Dropped: 5
Numerical: 9
Categorical: 6
Text: VIOLATION_DESCRIPTION


In [24]:
from sklearn.preprocessing import OneHotEncoder

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

print("Categorical encoder created.")

Categorical encoder created.


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

print("TF-IDF vectorizer created.")

TF-IDF vectorizer created.


In [26]:
# ============================================================
# FIT TF-IDF USING TRAINING DATA ONLY
# ============================================================

X_train_text = X_train[text_feature].fillna("")

tfidf_train = tfidf.fit_transform(X_train_text)

print("TF-IDF training matrix shape:", tfidf_train.shape)
print("Number of learned features:", len(tfidf.get_feature_names_out()))

TF-IDF training matrix shape: (81903, 5000)
Number of learned features: 5000


In [27]:
# ============================================================
# TRANSFORM TEST TEXT USING THE TRAINED TF-IDF
# ============================================================

X_test_text = X_test[text_feature].fillna("")

tfidf_test = tfidf.transform(X_test_text)

print("TF-IDF test matrix shape:", tfidf_test.shape)

TF-IDF test matrix shape: (20564, 5000)


In [28]:
# ============================================================
# FIT CATEGORICAL ENCODER ON TRAINING DATA
# ============================================================

X_train_cat = X_train[categorical_features].copy()
X_test_cat = X_test[categorical_features].copy()

# Convert to string so all categorical columns have a consistent type
X_train_cat = X_train_cat.astype(str)
X_test_cat = X_test_cat.astype(str)

# Fit ONLY on training data
cat_train = categorical_encoder.fit_transform(X_train_cat)

# Transform test using the same fitted encoder
cat_test = categorical_encoder.transform(X_test_cat)

print("Categorical training matrix:", cat_train.shape)
print("Categorical testing matrix: ", cat_test.shape)
print("Number of encoded categories:", cat_train.shape[1])

Categorical training matrix: (81903, 3011)
Categorical testing matrix:  (20564, 3011)
Number of encoded categories: 3011


In [29]:
# ============================================================
# NUMERICAL FEATURES
# ============================================================

X_train_num = X_train[numeric_features].astype(float).values
X_test_num = X_test[numeric_features].astype(float).values

print("Numerical training matrix:", X_train_num.shape)
print("Numerical testing matrix: ", X_test_num.shape)

Numerical training matrix: (81903, 9)
Numerical testing matrix:  (20564, 9)


In [30]:
from scipy.sparse import hstack, csr_matrix

# Combine numerical + categorical + text features
X_train_final = hstack([
    csr_matrix(X_train_num),
    cat_train,
    tfidf_train
], format="csr")

X_test_final = hstack([
    csr_matrix(X_test_num),
    cat_test,
    tfidf_test
], format="csr")

print("Final X_train shape:", X_train_final.shape)
print("Final X_test shape: ", X_test_final.shape)

Final X_train shape: (81903, 8020)
Final X_test shape:  (20564, 8020)


In [31]:
print("X_train_final:", X_train_final.shape)
print("y_train:", y_train.shape)
print("X_train_final dtype:", X_train_final.dtype)
print("Target values:")
print(y_train.value_counts())

X_train_final: (81903, 8020)
y_train: (81903,)
X_train_final dtype: float64
Target values:
SEVERITY
CLASS - 2    46320
CLASS - 1    32213
CLASS - 3     3370
Name: count, dtype: int64


In [32]:
try:
    baseline_model.fit(X_train_final, y_train)
    print("Baseline model trained successfully.")
except Exception as e:
    print(type(e).__name__)
    print(str(e))

NameError
name 'baseline_model' is not defined


In [33]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="saga",
    random_state=42
)

baseline_model.fit(X_train_final, y_train)

print("Baseline model trained successfully.")

Baseline model trained successfully.


c:\Users\gimha_9pk7du7\OneDrive\Desktop\projects\BuildSafe-AI\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [34]:
# ============================================================
# BASELINE PREDICTIONS
# ============================================================

y_pred = baseline_model.predict(X_test_final)

print("Predictions generated:", len(y_pred))
print("\nFirst 20 predictions:")
print(y_pred[:20])

Predictions generated: 20564

First 20 predictions:
['CLASS - 1' 'CLASS - 2' 'CLASS - 2' 'CLASS - 2' 'CLASS - 1' 'CLASS - 1'
 'CLASS - 2' 'CLASS - 2' 'CLASS - 2' 'CLASS - 1' 'CLASS - 3' 'CLASS - 2'
 'CLASS - 2' 'CLASS - 1' 'CLASS - 2' 'CLASS - 1' 'CLASS - 1' 'CLASS - 1'
 'CLASS - 3' 'CLASS - 3']


In [35]:
from sklearn.metrics import classification_report, accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Baseline Accuracy:", round(accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Baseline Accuracy: 0.8809

Classification Report:
              precision    recall  f1-score   support

   CLASS - 1       0.84      0.95      0.89      8460
   CLASS - 2       0.96      0.83      0.89     11287
   CLASS - 3       0.56      0.92      0.70       817

    accuracy                           0.88     20564
   macro avg       0.79      0.90      0.83     20564
weighted avg       0.90      0.88      0.88     20564



In [36]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["CLASS - 1", "CLASS - 2", "CLASS - 3"]
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual CLASS - 1", "Actual CLASS - 2", "Actual CLASS - 3"],
    columns=["Predicted CLASS - 1", "Predicted CLASS - 2", "Predicted CLASS - 3"]
)

print(cm_df)

                  Predicted CLASS - 1  Predicted CLASS - 2  \
Actual CLASS - 1                 8037                  369   
Actual CLASS - 2                 1434                 9325   
Actual CLASS - 3                   52                   12   

                  Predicted CLASS - 3  
Actual CLASS - 1                   54  
Actual CLASS - 2                  528  
Actual CLASS - 3                  753  


In [37]:
# ============================================================
# ERROR ANALYSIS — CLASS 2 PREDICTED AS CLASS 3
# ============================================================

error_mask = (
    (y_test == "CLASS - 2") &
    (y_pred == "CLASS - 3")
)

class2_as_class3 = test_df.loc[error_mask].copy()

print("CLASS 2 → CLASS 3 errors:", len(class2_as_class3))

print("\nViolation type:")
print(class2_as_class3["VIOLATION_TYPE"].value_counts())

print("\nHistorical violation count:")
print(class2_as_class3["PREVIOUS_VIOLATION_COUNT"].describe())

CLASS 2 → CLASS 3 errors: 528

Violation type:
VIOLATION_TYPE
Construction           280
Zoning                 135
Unknown                 72
Boilers                 30
Plumbing                 7
Quality of Life          3
Cranes and Derricks      1
Name: count, dtype: int64

Historical violation count:
count    528.000000
mean       0.274621
std        0.783099
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        5.000000
Name: PREVIOUS_VIOLATION_COUNT, dtype: float64


In [38]:
# ============================================================
# CLASS 3 VS FALSE CLASS 3 COMPARISON
# ============================================================

correct_class3 = test_df[
    (y_test == "CLASS - 3") &
    (y_pred == "CLASS - 3")
].copy()

false_class3 = test_df[
    (y_test == "CLASS - 2") &
    (y_pred == "CLASS - 3")
].copy()

print("Correct CLASS 3:", len(correct_class3))
print("False CLASS 3:", len(false_class3))

print("\n--- VIOLATION TYPE ---")

print("\nCorrect CLASS 3:")
print(correct_class3["VIOLATION_TYPE"].value_counts().head(10))

print("\nFalse CLASS 3:")
print(false_class3["VIOLATION_TYPE"].value_counts().head(10))

print("\n--- PREVIOUS VIOLATIONS ---")

print("\nCorrect CLASS 3:")
print(correct_class3["PREVIOUS_VIOLATION_COUNT"].describe())

print("\nFalse CLASS 3:")
print(false_class3["PREVIOUS_VIOLATION_COUNT"].describe())

Correct CLASS 3: 753
False CLASS 3: 528

--- VIOLATION TYPE ---

Correct CLASS 3:
VIOLATION_TYPE
Construction    540
Zoning          152
Unknown          35
Boilers          26
Name: count, dtype: int64

False CLASS 3:
VIOLATION_TYPE
Construction           280
Zoning                 135
Unknown                 72
Boilers                 30
Plumbing                 7
Quality of Life          3
Cranes and Derricks      1
Name: count, dtype: int64

--- PREVIOUS VIOLATIONS ---

Correct CLASS 3:
count    753.000000
mean       0.474104
std        1.130892
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        7.000000
Name: PREVIOUS_VIOLATION_COUNT, dtype: float64

False CLASS 3:
count    528.000000
mean       0.274621
std        0.783099
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        5.000000
Name: PREVIOUS_VIOLATION_COUNT, dtype: float64


In [39]:
print("Number of iterations used:", baseline_model.n_iter_)
print("Maximum iterations allowed:", baseline_model.max_iter)

Number of iterations used: [1000]
Maximum iterations allowed: 1000


In [40]:
baseline_model_converged = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    solver="saga",
    random_state=42
)

baseline_model_converged.fit(X_train_final, y_train)

print("Model trained.")
print("Iterations used:", baseline_model_converged.n_iter_)

Model trained.
Iterations used: [3000]


c:\Users\gimha_9pk7du7\OneDrive\Desktop\projects\BuildSafe-AI\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [41]:
# ============================================================
# MODEL 2 — DECISION TREE BASELINE
# ============================================================

from sklearn.tree import DecisionTreeClassifier

decision_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=20,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42
)

decision_tree.fit(X_train_final, y_train)

print("Decision Tree trained successfully.")
print("Tree depth:", decision_tree.get_depth())
print("Number of leaves:", decision_tree.get_n_leaves())

Decision Tree trained successfully.
Tree depth: 20
Number of leaves: 159


In [42]:
# ============================================================
# DECISION TREE — TEST PREDICTIONS
# ============================================================

y_pred_dt = decision_tree.predict(X_test_final)

print("Predictions generated:", len(y_pred_dt))

Predictions generated: 20564


In [43]:
# ============================================================
# DECISION TREE — EVALUATION
# ============================================================

from sklearn.metrics import accuracy_score, classification_report

dt_accuracy = accuracy_score(y_test, y_pred_dt)

print("Decision Tree Accuracy:", round(dt_accuracy, 4))

print("\nDecision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))

Decision Tree Accuracy: 0.8643

Decision Tree Classification Report:
              precision    recall  f1-score   support

   CLASS - 1       0.96      0.71      0.82      8460
   CLASS - 2       0.82      0.97      0.89     11287
   CLASS - 3       0.89      0.98      0.93       817

    accuracy                           0.86     20564
   macro avg       0.89      0.89      0.88     20564
weighted avg       0.88      0.86      0.86     20564



In [44]:
# ============================================================
# MODEL 3 — LINEAR SVM BASELINE
# ============================================================

from sklearn.svm import LinearSVC

linear_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=3000,
    random_state=42
)

linear_svm.fit(X_train_final, y_train)

print("Linear SVM trained successfully.")
print("Iterations used:", linear_svm.n_iter_)

Linear SVM trained successfully.
Iterations used: 35


In [45]:
# ============================================================
# LINEAR SVM — TEST PREDICTIONS
# ============================================================

y_pred_svm = linear_svm.predict(X_test_final)

print("Predictions generated:", len(y_pred_svm))

Predictions generated: 20564


In [46]:
# ============================================================
# LINEAR SVM — EVALUATION
# ============================================================

from sklearn.metrics import accuracy_score, classification_report

svm_accuracy = accuracy_score(y_test, y_pred_svm)

print("Linear SVM Accuracy:", round(svm_accuracy, 4))

print("\nLinear SVM Classification Report:")
print(classification_report(y_test, y_pred_svm))

Linear SVM Accuracy: 0.9938

Linear SVM Classification Report:
              precision    recall  f1-score   support

   CLASS - 1       0.99      0.99      0.99      8460
   CLASS - 2       1.00      0.99      0.99     11287
   CLASS - 3       1.00      1.00      1.00       817

    accuracy                           0.99     20564
   macro avg       1.00      1.00      1.00     20564
weighted avg       0.99      0.99      0.99     20564



In [47]:
# ============================================================
# LEAKAGE CHECK — IDENTICAL FEATURE RECORDS
# ============================================================

feature_cols = [c for c in train_df.columns if c != TARGET]

train_feature_keys = set(
    train_df[feature_cols].astype(str).agg("|".join, axis=1)
)

test_feature_keys = set(
    test_df[feature_cols].astype(str).agg("|".join, axis=1)
)

overlap = train_feature_keys.intersection(test_feature_keys)

print("Unique training feature records:", len(train_feature_keys))
print("Unique testing feature records:", len(test_feature_keys))
print("Identical feature records in both:", len(overlap))

Unique training feature records: 81451
Unique testing feature records: 20493
Identical feature records in both: 0


In [48]:
# ============================================================
# SEVERITY DISTRIBUTION BY INFRACTION CODE
# ============================================================

infraction_severity = pd.crosstab(
    df["INFRACTION_CODE1"],
    df["SEVERITY"],
    normalize="index"
)

print(infraction_severity.head(20))

SEVERITY          CLASS - 1  CLASS - 2  CLASS - 3
INFRACTION_CODE1                                 
1.00E+01                1.0        0.0        0.0
1.00E+02                1.0        0.0        0.0
1.00E+04                1.0        0.0        0.0
1.00E+05                1.0        0.0        0.0
1.00E+06                1.0        0.0        0.0
1.00E+07                1.0        0.0        0.0
1.00E+08                1.0        0.0        0.0
1.00E+09                1.0        0.0        0.0
101                     1.0        0.0        0.0
102                     1.0        0.0        0.0
103                     1.0        0.0        0.0
104                     1.0        0.0        0.0
106                     1.0        0.0        0.0
107                     1.0        0.0        0.0
108                     1.0        0.0        0.0
109                     1.0        0.0        0.0
110                     1.0        0.0        0.0
111                     1.0        0.0        0.0


In [49]:
# ============================================================
# INFRACTION CODE → SEVERITY PURITY
# ============================================================

infraction_counts = pd.crosstab(
    df["INFRACTION_CODE1"],
    df["SEVERITY"]
)

infraction_purity = (
    infraction_counts.max(axis=1) /
    infraction_counts.sum(axis=1)
)

print("Total unique infraction codes:",
      len(infraction_counts))

print("\nCodes appearing with only ONE severity class:",
      (infraction_purity == 1.0).sum())

print("\nPercentage of codes with 100% purity:",
      round((infraction_purity == 1.0).mean() * 100, 2), "%")

print("\nLowest purity codes:")
print(infraction_purity.sort_values().head(20))

Total unique infraction codes: 326

Codes appearing with only ONE severity class: 250

Percentage of codes with 100% purity: 76.69 %

Lowest purity codes:
INFRACTION_CODE1
379    0.500000
2K4    0.611111
236    0.666667
2L3    0.707317
276    0.750000
2P2    0.750000
222    0.777778
382    0.789474
274    0.800000
2D3    0.818182
2D9    0.833333
2L2    0.857143
233    0.875000
244    0.888889
2N8    0.897959
2Q1    0.900000
2N7    0.909091
2G3    0.916667
2N3    0.916667
212    0.921933
dtype: float64


In [50]:
# ============================================================
# MODEL 4 — MULTINOMIAL NAIVE BAYES
# ============================================================

from sklearn.naive_bayes import MultinomialNB

naive_bayes = MultinomialNB(
    alpha=1.0
)

naive_bayes.fit(X_train_final, y_train)

print("Multinomial Naive Bayes trained successfully.")

Multinomial Naive Bayes trained successfully.


In [51]:
# ============================================================
# MULTINOMIAL NAIVE BAYES — TEST PREDICTIONS
# ============================================================

y_pred_nb = naive_bayes.predict(X_test_final)

print("Predictions generated:", len(y_pred_nb))

Predictions generated: 20564


In [52]:
# ============================================================
# MULTINOMIAL NAIVE BAYES — EVALUATION
# ============================================================

from sklearn.metrics import accuracy_score, classification_report

nb_accuracy = accuracy_score(y_test, y_pred_nb)

print("Multinomial Naive Bayes Accuracy:", round(nb_accuracy, 4))

print("\nMultinomial Naive Bayes Classification Report:")
print(classification_report(y_test, y_pred_nb))

Multinomial Naive Bayes Accuracy: 0.9454

Multinomial Naive Bayes Classification Report:
              precision    recall  f1-score   support

   CLASS - 1       0.93      0.99      0.96      8460
   CLASS - 2       0.99      0.91      0.95     11287
   CLASS - 3       0.65      0.92      0.76       817

    accuracy                           0.95     20564
   macro avg       0.86      0.94      0.89     20564
weighted avg       0.95      0.95      0.95     20564



In [53]:
TARGET = "SEVERITY"

In [54]:
objects_to_check = [
    "linear_svm",
    "X_test_final",
    "y_test",
    "X_train_final",
    "y_train",
    "decision_tree",
    "naive_bayes"
]

for name in objects_to_check:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'MISSING'}")

linear_svm: AVAILABLE
X_test_final: AVAILABLE
y_test: AVAILABLE
X_train_final: AVAILABLE
y_train: AVAILABLE
decision_tree: AVAILABLE
naive_bayes: AVAILABLE


In [55]:
print("SVM predictions:", type(y_pred_svm))
print("Number of predictions:", len(y_pred_svm))
print("Test labels:", len(y_test))

SVM predictions: <class 'numpy.ndarray'>
Number of predictions: 20564
Test labels: 20564


In [56]:
print("TARGET:", TARGET)

print("\nOriginal training columns:")
print(train_df.columns.tolist())

print("\nNumber of final features:", X_train_final.shape[1])

print(
    "\nTarget present in training columns:",
    TARGET in train_df.columns
)

print(
    "Target present in X_train columns:",
    TARGET in X_train.columns
)

TARGET: SEVERITY

Original training columns:
['BIN', 'BORO', 'BLOCK', 'LOT', 'ISSUE_DATE', 'SEVERITY', 'VIOLATION_TYPE', 'RESPONDENT_CITY', 'RESPONDENT_ZIP', 'VIOLATION_DESCRIPTION', 'INFRACTION_CODE1', 'SECTION_LAW_DESCRIPTION1', 'ISSUE_DATE_DT', 'ISSUE_YEAR', 'ISSUE_MONTH', 'ISSUE_DAY', 'ISSUE_DAY_OF_WEEK', 'ISSUE_QUARTER', 'PREVIOUS_VIOLATION_COUNT', 'PREVIOUS_CLASS1_COUNT', 'PREVIOUS_CLASS2_COUNT', 'PREVIOUS_CLASS3_COUNT']

Number of final features: 8020

Target present in training columns: True
Target present in X_train columns: False


In [57]:
# ============================================================
# LINEAR SVM — CONFUSION MATRIX
# ============================================================

from sklearn.metrics import confusion_matrix
import pandas as pd

cm_svm = confusion_matrix(
    y_test,
    y_pred_svm,
    labels=["CLASS - 1", "CLASS - 2", "CLASS - 3"]
)

cm_svm_df = pd.DataFrame(
    cm_svm,
    index=[
        "Actual CLASS - 1",
        "Actual CLASS - 2",
        "Actual CLASS - 3"
    ],
    columns=[
        "Predicted CLASS - 1",
        "Predicted CLASS - 2",
        "Predicted CLASS - 3"
    ]
)

print(cm_svm_df)

                  Predicted CLASS - 1  Predicted CLASS - 2  \
Actual CLASS - 1                 8411                   49   
Actual CLASS - 2                   76                11210   
Actual CLASS - 3                    1                    0   

                  Predicted CLASS - 3  
Actual CLASS - 1                    0  
Actual CLASS - 2                    1  
Actual CLASS - 3                  816  


In [58]:
# ============================================================
# STAGE 7.1 — LINEAR SVM HYPERPARAMETER TUNING
# ============================================================

from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC

svm_tuning_model = LinearSVC(
    class_weight="balanced",
    max_iter=3000,
    random_state=42
)

param_grid = {
    "C": [0.1, 1.0, 10.0]
}

svm_grid = GridSearchCV(
    estimator=svm_tuning_model,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1
)

svm_grid.fit(X_train_final, y_train)

print("\nSVM tuning completed.")
print("Best C:", svm_grid.best_params_["C"])
print("Best CV Macro F1:", round(svm_grid.best_score_, 4))

Fitting 3 folds for each of 3 candidates, totalling 9 fits

SVM tuning completed.
Best C: 10.0
Best CV Macro F1: 0.9852


In [59]:
# ============================================================
# STAGE 7.2 — TUNED SVM TEST EVALUATION
# ============================================================

from sklearn.metrics import accuracy_score, classification_report

svm_best = svm_grid.best_estimator_

y_pred_svm_tuned = svm_best.predict(X_test_final)

tuned_svm_accuracy = accuracy_score(
    y_test,
    y_pred_svm_tuned
)

print("Tuned SVM Accuracy:",
      round(tuned_svm_accuracy, 4))

print("\nTuned SVM Classification Report:")
print(
    classification_report(
        y_test,
        y_pred_svm_tuned
    )
)

Tuned SVM Accuracy: 0.9939

Tuned SVM Classification Report:
              precision    recall  f1-score   support

   CLASS - 1       0.99      0.99      0.99      8460
   CLASS - 2       1.00      0.99      0.99     11287
   CLASS - 3       1.00      1.00      1.00       817

    accuracy                           0.99     20564
   macro avg       1.00      1.00      1.00     20564
weighted avg       0.99      0.99      0.99     20564



In [60]:
# ============================================================
# STAGE 7.3 — MULTINOMIAL NAIVE BAYES TUNING
# ============================================================

from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB

nb_tuning_model = MultinomialNB()

nb_param_grid = {
    "alpha": [0.01, 0.1, 1.0, 10.0]
}

nb_grid = GridSearchCV(
    estimator=nb_tuning_model,
    param_grid=nb_param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1
)

nb_grid.fit(X_train_final, y_train)

print("\nNaive Bayes tuning completed.")
print("Best alpha:", nb_grid.best_params_["alpha"])
print("Best CV Macro F1:", round(nb_grid.best_score_, 4))

Fitting 3 folds for each of 4 candidates, totalling 12 fits

Naive Bayes tuning completed.
Best alpha: 0.01
Best CV Macro F1: 0.9173


In [61]:
# ============================================================
# STAGE 7.4 — TUNED NAIVE BAYES TEST EVALUATION
# ============================================================

from sklearn.metrics import accuracy_score, classification_report

nb_best = nb_grid.best_estimator_

y_pred_nb_tuned = nb_best.predict(X_test_final)

tuned_nb_accuracy = accuracy_score(
    y_test,
    y_pred_nb_tuned
)

print("Tuned Naive Bayes Accuracy:",
      round(tuned_nb_accuracy, 4))

print("\nTuned Naive Bayes Classification Report:")
print(
    classification_report(
        y_test,
        y_pred_nb_tuned
    )
)

Tuned Naive Bayes Accuracy: 0.9507

Tuned Naive Bayes Classification Report:
              precision    recall  f1-score   support

   CLASS - 1       0.93      0.99      0.96      8460
   CLASS - 2       0.99      0.92      0.95     11287
   CLASS - 3       0.73      0.92      0.81       817

    accuracy                           0.95     20564
   macro avg       0.88      0.94      0.91     20564
weighted avg       0.96      0.95      0.95     20564



In [62]:
# ============================================================
# FINAL MODEL COMPARISON TABLE
# ============================================================

from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

comparison = []

models_results = [
    ("Logistic Regression", y_pred),
    ("Decision Tree", y_pred_dt),
    ("Linear SVM - Baseline", y_pred_svm),
    ("Naive Bayes - Baseline", y_pred_nb),
    ("Linear SVM - Tuned", y_pred_svm_tuned),
    ("Naive Bayes - Tuned", y_pred_nb_tuned)
]

for name, predictions in models_results:

    comparison.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro Precision": precision_score(
            y_test, predictions, average="macro"
        ),
        "Macro Recall": recall_score(
            y_test, predictions, average="macro"
        ),
        "Macro F1": f1_score(
            y_test, predictions, average="macro"
        ),
        "Class 3 F1": f1_score(
            y_test,
            predictions,
            labels=["CLASS - 3"],
            average=None
        )[0]
    })

comparison_df = pd.DataFrame(comparison)

print(
    comparison_df.round(4).to_string(index=False)
)

                 Model  Accuracy  Macro Precision  Macro Recall  Macro F1  Class 3 F1
   Logistic Regression    0.8809           0.7896        0.8993    0.8273      0.6998
         Decision Tree    0.8643           0.8914        0.8863    0.8793      0.9340
 Linear SVM - Baseline    0.9938           0.9951        0.9954    0.9953      0.9988
Naive Bayes - Baseline    0.9454           0.8568        0.9409    0.8902      0.7599
    Linear SVM - Tuned    0.9939           0.9952        0.9955    0.9953      0.9988
   Naive Bayes - Tuned    0.9507           0.8834        0.9447    0.9096      0.8137


In [63]:
# ============================================================
# FINAL SVM — ERROR ANALYSIS DATA
# ============================================================

error_mask = y_test != y_pred_svm_tuned

print("Total test records:", len(y_test))
print("Incorrect predictions:", error_mask.sum())
print(
    "Error rate:",
    round(error_mask.mean() * 100, 4),
    "%"
)

# Create error summary
error_summary = pd.DataFrame({
    "Actual": y_test[error_mask].values,
    "Predicted": y_pred_svm_tuned[error_mask]
})

print("\nError distribution:")
print(
    pd.crosstab(
        error_summary["Actual"],
        error_summary["Predicted"]
    )
)

Total test records: 20564
Incorrect predictions: 125
Error rate: 0.6079 %

Error distribution:
Predicted  CLASS - 1  CLASS - 2  CLASS - 3
Actual                                    
CLASS - 1          0         46          0
CLASS - 2         77          0          1
CLASS - 3          1          0          0


In [64]:
# ============================================================
# FINAL SVM — DETAILED ERROR ANALYSIS
# ============================================================

# Identify incorrect predictions
error_mask = y_test != y_pred_svm_tuned

# Get the corresponding test records
error_df = test_df.loc[y_test.index[error_mask]].copy()

# Add actual and predicted classes
error_df["ACTUAL_SEVERITY"] = y_test.loc[error_df.index]
error_df["PREDICTED_SEVERITY"] = y_pred_svm_tuned[
    error_mask
]

print("Error records:", len(error_df))

print("\n--- Error by Violation Type ---")
print(
    error_df["VIOLATION_TYPE"]
    .value_counts()
)

print("\n--- Error by Actual → Predicted ---")
print(
    pd.crosstab(
        error_df["ACTUAL_SEVERITY"],
        error_df["PREDICTED_SEVERITY"]
    )
)

Error records: 125

--- Error by Violation Type ---
VIOLATION_TYPE
Unknown                52
Construction           45
Plumbing                9
Elevators               8
Signs                   6
Cranes and Derricks     4
Zoning                  1
Name: count, dtype: int64

--- Error by Actual → Predicted ---
PREDICTED_SEVERITY  CLASS - 1  CLASS - 2  CLASS - 3
ACTUAL_SEVERITY                                    
CLASS - 1                   0         46          0
CLASS - 2                  77          0          1
CLASS - 3                   1          0          0


In [65]:
# ============================================================
# FINAL SVM — HISTORICAL FEATURE ANALYSIS OF ERRORS
# ============================================================

history_cols = [
    "PREVIOUS_VIOLATION_COUNT",
    "PREVIOUS_CLASS1_COUNT",
    "PREVIOUS_CLASS2_COUNT",
    "PREVIOUS_CLASS3_COUNT"
]

print("--- Historical features for error records ---")
print(error_df[history_cols].describe().round(3))

print("\n--- Historical features by actual severity ---")
print(
    error_df.groupby("ACTUAL_SEVERITY")[history_cols]
    .mean()
    .round(3)
)

--- Historical features for error records ---
       PREVIOUS_VIOLATION_COUNT  PREVIOUS_CLASS1_COUNT  PREVIOUS_CLASS2_COUNT  \
count                   125.000                125.000                 125.00   
mean                      3.048                  1.424                   1.60   
std                       5.550                  2.604                   3.51   
min                       0.000                  0.000                   0.00   
25%                       0.000                  0.000                   0.00   
50%                       1.000                  0.000                   0.00   
75%                       4.000                  2.000                   1.00   
max                      33.000                 12.000                  23.00   

       PREVIOUS_CLASS3_COUNT  
count                125.000  
mean                   0.024  
std                    0.154  
min                    0.000  
25%                    0.000  
50%                    0.000  
75%    

In [66]:
# ============================================================
# CHECK PREPROCESSING OBJECTS
# ============================================================

objects_to_check = [
    "svm_best",
    "tfidf_vectorizer",
    "categorical_encoder",
    "numeric_features",
    "categorical_features",
    "text_feature",
]

for name in objects_to_check:
    print(
        f"{name}: "
        f"{'AVAILABLE' if name in globals() else 'MISSING'}"
    )

svm_best: AVAILABLE
tfidf_vectorizer: MISSING
categorical_encoder: AVAILABLE
numeric_features: AVAILABLE
categorical_features: AVAILABLE
text_feature: AVAILABLE


In [68]:
# Find TF-IDF / vectorizer objects safely

global_items = list(globals().items())

for name, obj in global_items:
    if any(
        word in name.lower()
        for word in ["tfidf", "vectorizer", "tf_idf"]
    ):
        print(name, "->", type(obj))

TfidfVectorizer -> <class 'type'>
tfidf -> <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
tfidf_train -> <class 'scipy.sparse._csr.csr_matrix'>
tfidf_test -> <class 'scipy.sparse._csr.csr_matrix'>


In [69]:
# ============================================================
# SAVE FINAL MODEL + PREPROCESSING OBJECTS
# ============================================================

import os
import joblib

os.makedirs("models", exist_ok=True)

# Save final tuned SVM
joblib.dump(
    svm_best,
    "models/final_svm.pkl"
)

# Save fitted TF-IDF vectorizer
joblib.dump(
    tfidf,
    "models/tfidf_vectorizer.pkl"
)

# Save fitted categorical encoder
joblib.dump(
    categorical_encoder,
    "models/categorical_encoder.pkl"
)

# Save feature configuration
feature_config = {
    "text_feature": text_feature,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "n_tfidf_features": 5000,
    "n_categorical_features": 3011,
    "n_numeric_features": 9,
    "total_features": 8020
}

joblib.dump(
    feature_config,
    "models/feature_config.pkl"
)

print("Final model and preprocessing objects saved successfully.")

Final model and preprocessing objects saved successfully.
